1)
Датасет - Iris species.

2)
Размер - 150 строк.
Все признаки числовые.
Нет пропусков данных.
В датасете 50 строк с данными Iris-setosa, 50 с Iris-versicolor и 50 с Iris-virginica. Данные в датасете хорошо сбалансированы.

In [29]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.base import BaseEstimator
from sklearn import metrics
from sklearn.model_selection import GridSearchCV

df = pandas.read_csv("Iris.csv")
df.set_index("Id", inplace=True)
df

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
Id,,,,,
1,5.1,3.5,1.4,0.2,Iris-setosa
2,4.9,3.0,1.4,0.2,Iris-setosa
3,4.7,3.2,1.3,0.2,Iris-setosa
4,4.6,3.1,1.5,0.2,Iris-setosa
5,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...
146,6.7,3.0,5.2,2.3,Iris-virginica
147,6.3,2.5,5.0,1.9,Iris-virginica
148,6.5,3.0,5.2,2.0,Iris-virginica


In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 1 to 150
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   SepalLengthCm  150 non-null    float64
 1   SepalWidthCm   150 non-null    float64
 2   PetalLengthCm  150 non-null    float64
 3   PetalWidthCm   150 non-null    float64
 4   Species        150 non-null    str    
dtypes: float64(4), str(1)
memory usage: 6.0 KB


In [31]:
df.describe(include="all")

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
count,150.000000,150.000000,150.000000,150.000000,150
unique,NaN,NaN,NaN,NaN,3
top,NaN,NaN,NaN,NaN,Iris-setosa
freq,NaN,NaN,NaN,NaN,50
mean,5.843333,3.054000,3.758667,1.198667,NaN
std,0.828066,0.433594,1.764420,0.763161,NaN
min,4.300000,2.000000,1.000000,0.100000,NaN
25%,5.100000,2.800000,1.600000,0.300000,NaN
50%,5.800000,3.000000,4.350000,1.300000,NaN
75%,6.400000,3.300000,5.100000,1.800000,NaN


3.
Пропусков в датасете нет. Категориальный признак один, и это target столбец. Он не будет учавствовать в обучении, так что применим Label Encoding. ну и заодно сразу разделим датасет на Features и Target. Масштабировать ничего не надо, так как все признаки лежат примерно в одном диапазоне. 

In [32]:
labelencoder = LabelEncoder()
y = labelencoder.fit_transform(df["Species"])
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [33]:
X = df.copy().drop(["Species"], axis=1)
X

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
Id,,,,
1,5.1,3.5,1.4,0.2
2,4.9,3.0,1.4,0.2
3,4.7,3.2,1.3,0.2
4,4.6,3.1,1.5,0.2
5,5.0,3.6,1.4,0.2
...,...,...,...,...
146,6.7,3.0,5.2,2.3
147,6.3,2.5,5.0,1.9
148,6.5,3.0,5.2,2.0


Далее разделим данные на train, validation и test.

Выделение test:

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, train_size=0.8, random_state=0, stratify=y)

Выделение validation:

In [35]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, train_size=0.75, random_state=0, stratify=y_train)

Для kNN важно масштабирование, так как оно уравнивает важность признаков.

Если подбирать параметр на тестовой выборке, то можно переобучить модель. Это происходит потому что параметр подгоняется под определенные данные. Поэтому на test выборке нельзя подбирать параметры. А чтобы не подбирать их "на глаз" на тренировочной выборке, можно разделить датасет на train, validation и test, что я и сделал выше. Тогда можно на validation выборке чуть точнее подобрать параметр, а на test выборке проверить работоспособность модели. Однако и этого может быть мало, тогда можно использовать кросс-валидацию и прочее.

4.

In [36]:
class kNN(BaseEstimator):
    __metrics_list = ["euclidean", "manhattan", "minkowski", "cosine"]
    __weights_list = ["uniform", "distance"]

    def __init__(self, train_destination_array=None, train_category_array=None, n_neighbours=3, metric="euclidean", weights="uniform", *, p=3):
        self.__train_destination_array = np.asarray(train_destination_array)
        self.__train_category_array = np.asarray(train_category_array)
        
        if n_neighbours > 0:
            self.__n_neighbours = n_neighbours
        else:
            raise Exception("The number of the nearest neighbours can't be less than 1.")
        
        if metric in self.__metrics_list:
            self.__metric = metric
        else:
            raise Exception("There's no such metric.")
        
        if weights in self.__weights_list:
            self.__weights = weights
        else:
            raise Exception("You can use only \"unform\" or \"distance\" as weights.")
        
        if p >= 1:
            self.__p = p
        else:
            raise Exception("The parameter of the Minkowski metric can't be less than 1.")
        
    @property
    def train_destination_array(self):
        return self.__train_destination_array
        
    @train_destination_array.setter
    def train_destination_array(self, train_destination_array):
        self.__train_destination_array = train_destination_array

    @property
    def train_category_array(self):
        return self.__train_category_array
        
    @train_category_array.setter
    def train_category_array(self, train_category_array):
        self.__train_category_array = train_category_array

# check the shape of the arrays!!!!!!!!!!

    @property
    def n_neighbours(self):
        return self.__n_neighbours
    
    @n_neighbours.setter
    def n_neighbours(self, n_neighbours):
        if n_neighbours > 0:
            self.__n_neighbours = n_neighbours
        else: 
            raise Exception("The numbers of the nearest neighbours can't be less than 1.")
        
    @property
    def metric(self):
        return self.__metric
        
    @metric.setter
    def metric(self, metric):
        if metric in self.__metrics_list:
            self.__metric = metric
        else:
            raise Exception("There's no such metric.")
    
    @property
    def weights(self):
        return self.__weights
        
    @weights.setter
    def weights(self, weights):
        if weights in self.__weights_list:
            self.__weights = weights
        else:
            raise Exception("You can use only \"unform\" or \"distance\" as weights.")
        
    @property
    def p(self):
        return self.__p
        
    @p.setter
    def p(self, p):
        if p >= 1:
            self.__p = p
        else:
            raise Exception("The parameter of the Minkowski metric can't be less than 1.")
        
    def destination(self, target_array):
        target_array = np.asarray(target_array)
        if self.__metric == "euclidean":
            return np.sqrt(np.sum((target_array[:, np.newaxis, :] - self.__train_destination_array[np.newaxis, :, :]) ** 2, axis = 2))
        elif self.__metric == "manhattan":
            return np.sum(np.abs((target_array[:, np.newaxis, :] - self.__train_destination_array[np.newaxis, :, :])), axis=2)
        elif self.__metric == "minkowski":
            return np.sum(np.abs(target_array[:, np.newaxis, :] - self.__train_destination_array[np.newaxis, :, :]) ** self.__p, axis = 2) ** (1/self.__p)
        elif self.__metric == "cosine":
            return 1 - ((target_array @ self.__train_destination_array.T) / ((np.sqrt(np.sum(target_array**2, axis=1) + 1e-10))[:, np.newaxis] * (np.sqrt(np.sum(self.__train_destination_array**2, axis=1) + 1e-10))[np.newaxis, :]))
        else:
            raise ValueError("Unknown metric")
        
    def fit(self, train_destination_array, train_category_array):
        self.__train_destination_array = np.asarray(train_destination_array)
        self.__train_category_array = np.asarray(train_category_array)
        return self
        
    def predict(self, target_array):
        if self.__weights=="uniform":
            target_array = np.asarray(target_array)
            D = self.destination(target_array)
            idx = np.argsort(D)
            y_nearest = np.take_along_axis(self.__train_category_array[np.newaxis, :], idx)
            if self.__n_neighbours < y_nearest.shape[1]:
                y_nearest = y_nearest[:, :self.__n_neighbours]
            return np.apply_along_axis(lambda row: np.bincount(row).argmax(), axis=1, arr=y_nearest)
        elif self.__weights=="distance":
            target_array = np.asarray(target_array)
            D = self.destination(target_array)
            idx = np.argsort(D)
            y_nearest = np.take_along_axis(self.__train_category_array[np.newaxis, :], idx)
            D = np.take_along_axis(self.__train_category_array[np.newaxis, :], idx)
            D = 1 / (D + 1e-10)
            if self.__n_neighbours < y_nearest.shape[1]:
                y_nearest = y_nearest[:, :self.__n_neighbours]
                D = D[:, :self.__n_neighbours]
            y_pred = np.empty(D.shape[0])
            for i in range(0, D.shape[0]):
                res = pandas.DataFrame({'key': y_nearest[i], 'val': D[i]}).groupby('key')['val'].sum().to_dict()
                y_pred[i] = max(res, key=res.get)
            return y_pred



In [53]:
knn = kNN(X_train, y_train, 20, metric = "cosine", weights="uniform", p=100)
R = knn.predict(X_test)
R

array([0, 1, 0, 2, 0, 1, 2, 0, 0, 1, 2, 1, 1, 2, 1, 2, 2, 1, 1, 0, 0, 2,
       2, 2, 0, 1, 1, 2, 0, 0])

In [54]:
y_test

array([0, 1, 0, 2, 0, 1, 2, 0, 0, 1, 2, 1, 1, 2, 1, 2, 2, 1, 1, 0, 0, 2,
       2, 2, 0, 1, 1, 2, 0, 0])

In [55]:
print(metrics.accuracy_score(y_test, R))

1.0


In [56]:
print(metrics.f1_score(R, y_test, average=None))

[1. 1. 1.]


In [41]:
knn.get_params()

{'metric': 'cosine',
 'n_neighbours': 15,
 'p': 100,
 'train_category_array': array([0, 1, 1, 1, 2, 2, 0, 0, 0, 0, 2, 2, 0, 0, 2, 0, 1, 0, 2, 2, 1, 0,
        2, 1, 2, 2, 1, 0, 1, 0, 0, 2, 0, 2, 2, 0, 2, 2, 0, 2, 2, 2, 1, 0,
        1, 1, 2, 0, 2, 1, 2, 2, 1, 1, 0, 2, 0, 1, 0, 1, 1, 2, 1, 2, 0, 1,
        0, 1, 1, 1, 2, 0, 1, 0, 1, 2, 1, 1, 0, 0, 1, 0, 1, 2, 2, 1, 2, 0,
        1, 0]),
 'train_destination_array': array([[5.7, 3.8, 1.7, 0.3],
        [6.6, 3. , 4.4, 1.4],
        [5.4, 3. , 4.5, 1.5],
        [6. , 3.4, 4.5, 1.6],
        [7.9, 3.8, 6.4, 2. ],
        [6. , 2.2, 5. , 1.5],
        [4.4, 2.9, 1.4, 0.2],
        [4.8, 3.1, 1.6, 0.2],
        [5. , 3.5, 1.6, 0.6],
        [5.4, 3.7, 1.5, 0.2],
        [6.7, 3.3, 5.7, 2.5],
        [6.3, 2.5, 5. , 1.9],
        [5.4, 3.4, 1.7, 0.2],
        [5.1, 3.5, 1.4, 0.3],
        [7.2, 3.6, 6.1, 2.5],
        [5.4, 3.4, 1.5, 0.4],
        [6.8, 2.8, 4.8, 1.4],
        [5. , 3.3, 1.4, 0.2],
        [6.5, 3. , 5.5, 1.8],
        [6.5, 

In [ ]:
params = [{
    'metric': ['euclidean', 'manhattan', 'cosine'],
    'n_neighbours': list(range(1, 50)),
    'weights': ['uniform', 'distance']
},
{
    'metric': ['minkowski'],
    'n_neighbours': list(range(1, 50)),
    'p': list(range(3, 15)),
    'weights': ['uniform', 'distance']
}
]

gs = GridSearchCV(kNN(), params, scoring='f1_macro', cv=5, n_jobs=10)
gs.fit(X_train, y_train)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",kNN(train_cat...dtype=object))
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'metric': ['euclidean', 'manhattan', ...], 'n_neighbours': [1, 2, ...], 'weights': ['uniform', 'distance']}, {'metric': ['minkowski'], 'n_neighbours': [1, 2, ...], 'p': [3, 4, ...], 'weights': ['uniform', 'distance']}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intContro

In [ ]:
results = pandas.DataFrame(gs.cv_results_)

In [65]:
minkowski_params_results = results[(results['param_metric'] == 'euclidean') & (results['param_weights'] == 'uniform')]
minkowski_params_results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_metric,param_n_neighbours,param_weights,param_p,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.001434,0.000168,0.003908,0.000496,euclidean,1,uniform,NaN,"{'metric': 'euclidean', 'n_neighbours': 1, 'we...",0.925926,0.822222,0.910714,0.940741,0.885714,0.897063,0.041610,470
2,0.001077,0.000079,0.003815,0.000193,euclidean,2,uniform,NaN,"{'metric': 'euclidean', 'n_neighbours': 2, 'we...",0.850000,0.771429,0.821429,0.940741,0.940741,0.864868,0.066862,723
4,0.000892,0.000151,0.003375,0.000661,euclidean,3,uniform,NaN,"{'metric': 'euclidean', 'n_neighbours': 3, 'we...",0.850000,0.932773,0.821429,0.940741,0.885714,0.886131,0.046151,567
6,0.000736,0.000101,0.002639,0.000233,euclidean,4,uniform,NaN,"{'metric': 'euclidean', 'n_neighbours': 4, 'we...",0.850000,0.875000,0.821429,0.940741,0.940741,0.885582,0.048122,593
8,0.000712,0.000076,0.002543,0.000181,euclidean,5,uniform,NaN,"{'metric': 'euclidean', 'n_neighbours': 5, 'we...",1.000000,0.875000,0.821429,0.940741,0.940741,0.915582,0.061488,67
10,0.000636,0.000021,0.002436,0.000035,euclidean,6,uniform,NaN,"{'metric': 'euclidean', 'n_neighbours': 6, 'we...",0.925926,0.875000,0.821429,0.940741,0.940741,0.900767,0.046448,385
12,0.000649,0.000123,0.002818,0.000291,euclidean,7,uniform,NaN,"{'metric': 'euclidean', 'n_neighbours': 7, 'we...",0.925926,0.875000,0.866667,0.940741,0.940741,0.909815,0.032392,189
14,0.000754,0.000139,0.003319,0.000569,euclidean,8,uniform,NaN,"{'metric': 'euclidean', 'n_neighbours': 8, 'we...",0.925926,0.940741,0.821429,0.940741,0.940741,0.913915,0.046598,97
16,0.000691,0.000149,0.003052,0.000517,euclidean,9,uniform,NaN,"{'metric': 'euclidean', 'n_neighbours': 9, 'we...",0.925926,0.940741,0.821429,0.940741,0.940741,0.913915,0.046598,97
18,0.000760,0.000087,0.003353,0.000203,euclidean,10,uniform,NaN,"{'metric': 'euclidean', 'n_neighbours': 10, 'w...",0.925926,0.940741,0.821429,0.940741,0.940741,0.913915,0.046598,97


5.

In [63]:
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_metric,param_n_neighbours,param_weights,param_p,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.001434,0.000168,0.003908,0.000496,euclidean,1,uniform,NaN,"{'metric': 'euclidean', 'n_neighbours': 1, 'we...",0.925926,0.822222,0.910714,0.940741,0.885714,0.897063,0.041610,470
1,0.001840,0.000161,0.022275,0.001043,euclidean,1,distance,NaN,"{'metric': 'euclidean', 'n_neighbours': 1, 'we...",0.925926,0.822222,0.910714,0.940741,0.885714,0.897063,0.041610,470
2,0.001077,0.000079,0.003815,0.000193,euclidean,2,uniform,NaN,"{'metric': 'euclidean', 'n_neighbours': 2, 'we...",0.850000,0.771429,0.821429,0.940741,0.940741,0.864868,0.066862,723
3,0.001032,0.000162,0.021519,0.002036,euclidean,2,distance,NaN,"{'metric': 'euclidean', 'n_neighbours': 2, 'we...",0.850000,0.771429,0.821429,0.940741,0.940741,0.864868,0.066862,723
4,0.000892,0.000151,0.003375,0.000661,euclidean,3,uniform,NaN,"{'metric': 'euclidean', 'n_neighbours': 3, 'we...",0.850000,0.932773,0.821429,0.940741,0.885714,0.886131,0.046151,567
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,0.000693,0.000106,0.018836,0.001302,minkowski,49,distance,12.0,"{'metric': 'minkowski', 'n_neighbours': 49, 'p...",0.538462,0.166667,0.095238,0.166667,0.166667,0.226740,0.158297,1456
1466,0.000664,0.000097,0.003289,0.001296,minkowski,49,uniform,13.0,"{'metric': 'minkowski', 'n_neighbours': 49, 'p...",0.390351,0.500000,0.452381,0.657143,0.585185,0.517012,0.094651,1247
1467,0.000679,0.000102,0.018021,0.001476,minkowski,49,distance,13.0,"{'metric': 'minkowski', 'n_neighbours': 49, 'p...",0.538462,0.166667,0.095238,0.166667,0.166667,0.226740,0.158297,1456
1468,0.000685,0.000092,0.002808,0.000523,minkowski,49,uniform,14.0,"{'metric': 'minkowski', 'n_neighbours': 49, 'p...",0.390351,0.500000,0.452381,0.657143,0.585185,0.517012,0.094651,1247


In [62]:
results[(results['param_metric'] == 'cosine') & (results['param_weights'] == 'uniform') & (results['param_n_neighbours'] == 20)]

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_metric,param_n_neighbours,param_weights,param_p,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
234,0.000635,0.000137,0.002947,0.000756,cosine,20,uniform,NaN,"{'metric': 'cosine', 'n_neighbours': 20, 'weig...",1.0,1.0,1.0,0.940741,0.940741,0.976296,0.029031,1
